# Sakina — Brain Response to Quran Recitation

Visualize how the brain responds to Surah Al-Fatiha using Meta's TRIBE v2 model.

**Setup**: Run the first cell, restart runtime, then run all cells.

In [ ]:
!pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

In [ ]:
from tribev2.demo_utils import TribeModel
from tribev2.plotting import PlotBrain
from pathlib import Path
import numpy as np

CACHE = Path('./cache')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE)
plotter = PlotBrain(mesh='fsaverage5')
print('Model loaded!')

In [ ]:
# Download Al-Fatiha recitation
import urllib.request
CACHE.mkdir(exist_ok=True)
url = 'https://download.quranicaudio.com/qdc/mishari_al_afasy/murattal/1.mp3'
audio_path = CACHE / 'al-fatiha.mp3'
if not audio_path.exists():
    urllib.request.urlretrieve(url, audio_path)
print(f'Audio: {audio_path} ({audio_path.stat().st_size / 1024:.0f} KB)')

In [ ]:
# Process audio and predict brain response
import pandas as pd
from tribev2.demo_utils import get_audio_and_text_events

event = {'type': 'Audio', 'filepath': str(audio_path), 'start': 0, 'timeline': 'default', 'subject': 'default'}
events = get_audio_and_text_events(pd.DataFrame([event]), audio_only=True)
print(f'Events: {len(events)} rows')

preds, segments = model.predict(events=events)
print(f'Predictions: {preds.shape} ({preds.shape[0]} timesteps × {preds.shape[1]} vertices)')

In [ ]:
# Visualize brain response — left hemisphere
fig = plotter.plot_timesteps(preds[:15], cmap='fire', norm_percentile=99, vmin=0.3, views='left')
fig.suptitle('Sakina — Al-Fatiha Brain Response (Left)', fontsize=14)
fig

In [ ]:
# Right hemisphere
fig2 = plotter.plot_timesteps(preds[:15], cmap='fire', norm_percentile=99, vmin=0.3, views='right')
fig2.suptitle('Sakina — Al-Fatiha Brain Response (Right)', fontsize=14)
fig2

In [ ]:
# Top view
fig3 = plotter.plot_timesteps(preds[:15], cmap='fire', norm_percentile=99, vmin=0.3, views='dorsal')
fig3.suptitle('Sakina — Al-Fatiha Brain Response (Top)', fontsize=14)
fig3